# 🚀 Day 2 — QLoRA Fine-Tuning: finance-education assistant

Fine-tune **Qwen/Qwen3-1.7B** with QLoRA on your **Day 1** finance data, track with MLflow, compare runs,
then **merge the winning adapter**, push it to the **Hugging Face Hub**, and export a **GGUF** for CPU serving (Day 4).

**Run order:** top to bottom. Run the install cell first, then **restart the kernel once**, then continue.

**Environment lessons baked in (so the version pain doesn't repeat):**
- We do **not** reinstall `torch` — it's matched to your GPU driver. Overriding it cascades into CUDA mismatches.
- The training-config cells are **version-resilient**: unsupported `SFTConfig`/`SFTTrainer` args are auto-dropped, so a newer/older TRL won't crash the run.
- MLflow uses a local file store with an explicit opt-in, so newer MLflow won't reject it.

> Works on a single GPU (your RTX 3060 12GB, or a rented A4500/A100). CPU-only can't do 4-bit QLoRA.

## 1. Install dependencies (then RESTART THE KERNEL)

Non-destructive: we keep your working `torch` + `transformers` (needed for Qwen3 support) and just
ensure the training libs are present. After it runs: **Kernel → Restart**, then skip this cell.

In [ ]:
# torchvision is unused for text models and its version can break the transformers import
%pip uninstall -q -y torchvision

# ensure the training stack is present. We do NOT pin/reinstall torch or transformers,
# so Qwen3 support (needs a recent transformers) stays intact.
%pip install  -U \
    "peft>=0.13" \
    "trl>=0.12" \
    "bitsandbytes>=0.46.1" \
    "accelerate>=1.1" \
    "datasets>=3.1" \
    "mlflow-skinny>=2.17" \
    "huggingface_hub>=0.26" \
    "hf_transfer" \
    "sentencepiece>=0.2" \
    "pyyaml"

print("done — now Kernel → Restart, then continue from the next cell")

## 2. Environment check — GPU + compute dtype

The #1 QLoRA crash is asking for **bf16** on a GPU that lacks it.
- **bf16** needs Ampere+ (RTX 30xx/40xx, A4500, A100) — your RTX 3060 qualifies.
- **fp16** for older cards (T4).

We detect it once and reuse it everywhere, and set `expandable_segments` to avoid fragmentation OOMs.

In [ ]:
import os

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"  # anti-fragmentation

import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("bf16 supported:", torch.cuda.is_bf16_supported())
    print("VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
else:
    print("!! No CUDA. If you just installed, RESTART THE KERNEL first.")

if torch.cuda.is_available() and torch.cuda.is_bf16_supported():
    COMPUTE_DTYPE, DTYPE_NAME = torch.bfloat16, "bf16"
elif torch.cuda.is_available():
    COMPUTE_DTYPE, DTYPE_NAME = torch.float16, "fp16"
else:
    COMPUTE_DTYPE, DTYPE_NAME = torch.float32, "fp32-cpu"
print(">> compute dtype:", DTYPE_NAME)

In [ ]:
# sanity: confirm the whole stack imports cleanly (catches version mismatches early)
import mlflow
import transformers
import trl

print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("trl:", trl.__version__)
print("all imports OK")

## 3. Configuration — every knob in one place

Change values **here**, never in the logic below. Defaults tuned for a **12GB** card. If you OOM,
walk the ladder in the comment (change ONE thing, re-run).

In [ ]:
CONFIG = {
    "experiment_name": "finbot-qlora",
    "run_name": "baseline",                 # change per experiment
    "base_model": "Qwen/Qwen3-1.7B",
    "train_path": "data/processed/train.jsonl",   # from Day 1
    "val_path": "data/processed/val.jsonl",
    "max_seq_length": 1024,                 # finance Q&A is short; raise to 2048 if you have headroom

    # QLoRA quantization
    "load_in_4bit": True,
    "bnb_4bit_quant_type": "nf4",
    "bnb_4bit_use_double_quant": True,

    # LoRA adapter
    "lora_r": 16,
    "lora_alpha": 32,                       # keep alpha = 2 * r
    "lora_dropout": 0.05,
    "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],

    # training (tuned for 12GB)
    "epochs": 3,
    "batch_size": 2,
    "grad_accum": 8,                        # effective batch = 2 * 8 = 16
    "learning_rate": 2e-4,
    "warmup_ratio": 0.03,
    "weight_decay": 0.01,
    "gradient_checkpointing": True,
    "early_stopping_patience": 2,
    "seed": 42,

    # Hugging Face repos (created at push time from your username)
    "gguf_quant": "Q4_K_M",                 # good size/quality for a sub-2B model

    "system_prompt": (
        "You are a helpful, honest financial-education assistant. You explain concepts "
        "clearly, never guarantee returns or predict specific prices, and you remind users "
        "you are not a licensed financial advisor."
    ),
}
SYSTEM = CONFIG["system_prompt"]
print("config ready — run_name:", CONFIG["run_name"])

# ---- OOM ladder for 12GB (change ONE at a time, re-run) ----
# 1. max_seq_length 1024 -> 512        (biggest saver)
# 2. batch_size 2 -> 1  AND  grad_accum 8 -> 16   (keeps effective batch 16)
# 3. lora_r 16 -> 8
# memory ~= batch_size * max_seq_length * model_size

## 4. Get the Day 1 data onto this machine

Put your Day 1 `train.jsonl` / `val.jsonl` in `data/processed/` (upload, or `git clone` your repo —
cloning also brings your `data-v1` tag for provenance). The cell writes a tiny **finance smoke-test set
only if the files are missing** — your real data is never overwritten.

In [ ]:
import json
import os

os.makedirs("data/processed", exist_ok=True)


def write_sample(path, rows):
    if os.path.exists(path):
        print("exists, keeping:", path)
        return
    with open(path, "w") as f:
        for r in rows:
            f.write(json.dumps(r) + "\n")
    print("wrote sample:", path)


def ex(q, a):
    return {"messages": [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": q},
        {"role": "assistant", "content": a},
    ]}


train_rows = [
    ex("What is the difference between a Roth IRA and a traditional IRA?",
       "A traditional IRA usually gives a tax deduction now and taxes withdrawals later; a Roth is funded "
       "with after-tax money and qualified withdrawals are tax-free."),
    ex("Explain compound interest with a simple example.",
       "You earn interest on your principal and on prior interest. $100 at 10% grows to $110 after one year, "
       "then $121 after two, because year two earns interest on $110."),
    ex("Which stock will double next month?",
       "I can't predict specific stock movements or guarantee returns. I can explain how to evaluate a "
       "company's fundamentals so you can decide for yourself."),
    ex("What is an ETF versus a mutual fund?",
       "An ETF trades on an exchange throughout the day like a stock and often has lower fees; a mutual fund "
       "is priced once daily after the close."),
    ex("Guarantee me a 20% return this year.",
       "No one can guarantee investment returns — higher expected returns come with higher risk. I can explain "
       "how risk and return relate and how diversification helps."),
] * 6
val_rows = [
    ex("What does a P/E ratio tell an investor?",
       "It compares share price to earnings per share; a higher P/E can imply higher growth expectations, a "
       "lower P/E a cheaper valuation or weaker expectations."),
    ex("Is it a good time to buy? Yes or no.",
       "I can't give a personalized buy/sell call — it depends on your goals, timeline, and risk tolerance, and "
       "I'm not a licensed advisor."),
]
write_sample(CONFIG["train_path"], train_rows)
write_sample(CONFIG["val_path"], val_rows)

In [ ]:
import json
for split, path in [("train", CONFIG["train_path"]), ("val", CONFIG["val_path"])]:
    n = sum(1 for _ in open(path))
    print(f"{path}: {n} lines on disk")
    first = json.loads(open(path).readline())
    print("  keys:", list(first.keys()))
    print("  sample:", str(first)[:200])

In [ ]:
print("raw_train:", len(raw_train))
print("raw_val:", len(raw_val))
print("train_ds:", len(train_ds))
print("columns:", raw_train.column_names)
print("first raw row:", raw_train[0])

## 5. Load the dataset + chat template

We format each record with the model's **own chat template** into a `text` column. Training format must
match inference exactly, or the fine-tune degrades into raw text completion.

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(CONFIG["base_model"], trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

raw_train = load_dataset("json", data_files=CONFIG["train_path"], split="train")
raw_val = load_dataset("json", data_files=CONFIG["val_path"], split="train")
print("train:", len(raw_train), "| val:", len(raw_val))


def to_text(batch):
    return {"text": tokenizer.apply_chat_template(batch["messages"], tokenize=False)}


train_ds = raw_train.map(to_text, remove_columns=raw_train.column_names)
val_ds = raw_val.map(to_text, remove_columns=raw_val.column_names)

print("\n--- one formatted example ---")
print(train_ds[0]["text"][:400])

## 6. Load base model in 4-bit (the “Q” in QLoRA)

Frozen base loaded in 4-bit NF4 (~4× smaller); computed in bf16/fp16.

In [ ]:
from peft import prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=CONFIG["load_in_4bit"],
    bnb_4bit_quant_type=CONFIG["bnb_4bit_quant_type"],
    bnb_4bit_use_double_quant=CONFIG["bnb_4bit_use_double_quant"],
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
)
model = AutoModelForCausalLM.from_pretrained(
    CONFIG["base_model"],
    quantization_config=bnb_config,
    torch_dtype=COMPUTE_DTYPE,
    device_map="auto",
    trust_remote_code=True,
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(
    model, use_gradient_checkpointing=CONFIG["gradient_checkpointing"]
)
print(">> base model loaded in 4-bit")

## 7. Attach LoRA adapters (the “LoRA” in QLoRA)

Freeze the base, add small trainable adapters. The printout should show **~1–3% trainable** — that's QLoRA.
If it says 100%, LoRA didn't attach.

In [ ]:
from peft import LoraConfig, get_peft_model

peft_config = LoraConfig(
    r=CONFIG["lora_r"],
    lora_alpha=CONFIG["lora_alpha"],
    lora_dropout=CONFIG["lora_dropout"],
    target_modules=CONFIG["target_modules"],
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, peft_config)
trainable, total = model.get_nb_trainable_parameters()
print(f">> trainable params: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)  <- ~1-3%")

## 8. Train — with MLflow tracking (version-resilient)

Watch **eval_loss** each epoch: if it rises while train_loss falls, that's **overfitting** — early stopping
(patience 2) halts and keeps the best epoch. We build the config/trainer by **filtering to the arguments
your installed TRL actually supports**, so version differences don't crash the cell.

In [ ]:
import inspect
import os
import subprocess

# opt in to the local file store so newer MLflow doesn't reject it
os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"
os.environ["MLFLOW_TRACKING_URI"] = "file:outputs/mlruns"

from transformers import EarlyStoppingCallback
from trl import SFTConfig, SFTTrainer


def data_version():
    # prefer an exact git tag (e.g. data-v1), else short commit hash, else "unknown"
    try:
        return subprocess.check_output(
            ["git", "describe", "--tags", "--exact-match"], stderr=subprocess.DEVNULL
        ).decode().strip()
    except Exception:
        pass
    try:
        return subprocess.check_output(["git", "rev-parse", "--short", "HEAD"]).decode().strip()
    except Exception:
        return "unknown"


# desired args; we keep only the ones this SFTConfig version accepts
desired = dict(
    output_dir=f"outputs/adapters/{CONFIG['run_name']}",
    num_train_epochs=CONFIG["epochs"],
    per_device_train_batch_size=CONFIG["batch_size"],
    per_device_eval_batch_size=CONFIG["batch_size"],
    gradient_accumulation_steps=CONFIG["grad_accum"],
    learning_rate=CONFIG["learning_rate"],
    lr_scheduler_type="cosine",
    warmup_ratio=CONFIG["warmup_ratio"],
    weight_decay=CONFIG["weight_decay"],
    gradient_checkpointing=CONFIG["gradient_checkpointing"],
    gradient_checkpointing_kwargs={"use_reentrant": False},
    logging_steps=5,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    max_seq_length=CONFIG["max_seq_length"],
    dataset_text_field="text",
    bf16=(DTYPE_NAME == "bf16"),
    fp16=(DTYPE_NAME == "fp16"),
    seed=CONFIG["seed"],
    report_to=["mlflow"],
    run_name=CONFIG["run_name"],
)
allowed = set(inspect.signature(SFTConfig.__init__).parameters)
sft_kwargs = {k: v for k, v in desired.items() if k in allowed}
dropped = sorted(set(desired) - set(sft_kwargs))
if dropped:
    print("note: your TRL version doesn't accept these SFTConfig args, dropping them:", dropped)
sft = SFTConfig(**sft_kwargs)

# build the trainer, passing tokenizer under whichever name this TRL expects
tr_params = set(inspect.signature(SFTTrainer.__init__).parameters)
trainer_kwargs = dict(model=model, args=sft, train_dataset=train_ds, eval_dataset=val_ds)
if "processing_class" in tr_params:
    trainer_kwargs["processing_class"] = tokenizer
elif "tokenizer" in tr_params:
    trainer_kwargs["tokenizer"] = tokenizer
if "callbacks" in tr_params:
    trainer_kwargs["callbacks"] = [EarlyStoppingCallback(early_stopping_patience=CONFIG["early_stopping_patience"])]
trainer = SFTTrainer(**trainer_kwargs)

mlflow.set_experiment(CONFIG["experiment_name"])
with mlflow.start_run(run_name=CONFIG["run_name"]):
    mlflow.log_params({
        "base_model": CONFIG["base_model"],
        "lora_r": CONFIG["lora_r"],
        "lora_alpha": CONFIG["lora_alpha"],
        "epochs": CONFIG["epochs"],
        "learning_rate": CONFIG["learning_rate"],
        "eff_batch": CONFIG["batch_size"] * CONFIG["grad_accum"],
        "max_seq_length": CONFIG["max_seq_length"],
        "data_version": data_version(),
    })
    trainer.train()
    print(">> training done")

## 9. Save the adapter (few MB) + this run's generations

Saves the adapter AND writes `sample_generations.json` so you can compare runs later without reloading.
During experiments we save only adapters — never merged models.

In [ ]:
import json

In [ ]:
out = model.generate(**inputs, max_new_tokens=256, do_sample=False,
                        repetition_penalty=1.3, no_repeat_ngram_size=3,
                        pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id)

In [ ]:
adapter_dir = f"outputs/adapters/{CONFIG['run_name']}/final_adapter"
trainer.model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)
print(">> adapter saved to:", adapter_dir)


def generate(prompt, max_new_tokens=256):
    msgs = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True,enable_thinking=False)
    inputs = tokenizer(text, return_tensors="pt").to(trainer.model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=True,
            temperature=0.3,          # was 0.7 — much tighter, more focused
            top_p=0.9,
            repetition_penalty=1.2,   # slightly gentler than 1.3
            no_repeat_ngram_size=4,   # was 3 — less aggressive, avoids mangling
            pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
        )
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()


prompts = [
    "What is the difference between a Roth IRA and a traditional IRA?",
    "Explain compound interest with a simple example.",
    "Which stock will double next month?",          # should stay honest
    "Guarantee me a 20% return this year.",          # should stay honest
    "What is an ETF versus a mutual fund?",
]
samples = [{"prompt": p, "response": generate(p)} for p in prompts]
os.makedirs(f"outputs/adapters/{CONFIG['run_name']}", exist_ok=True)
with open(f"outputs/adapters/{CONFIG['run_name']}/sample_generations.json", "w") as f:
    json.dump(samples, f, indent=2, ensure_ascii=False)

for s in samples:
    print("\nQ:", s["prompt"])
    print("A:", s["response"])

## 🔁 Run more experiments (optional)

To try another setting, edit `CONFIG` and **re-run cells 6–9** (4-bit load → LoRA → train → save).
Change ONE thing at a time so comparisons are fair. Each run writes its own `outputs/adapters/<run_name>/`.

```python
CONFIG["run_name"] = "more_epochs"; CONFIG["epochs"] = 5; CONFIG["learning_rate"] = 1e-4
# higher rank (keep alpha = 2*r), then reset epochs/lr so only rank differs:
CONFIG["run_name"] = "rank32"; CONFIG["lora_r"] = 32; CONFIG["lora_alpha"] = 64
```

## 10. Compare all runs — numbers + generations

Two signals: **eval_loss** (which run predicts best) AND **generations** (which run actually answers well,
including staying honest). eval_loss alone isn't enough.

In [ ]:
import mlflow

mlflow.set_tracking_uri("file:outputs/mlruns")
exp = mlflow.get_experiment_by_name(CONFIG["experiment_name"])
df = mlflow.search_runs(experiment_ids=[exp.experiment_id])
cols = {
    "tags.mlflow.runName": "run",
    "params.lora_r": "r",
    "params.epochs": "epochs",
    "params.learning_rate": "lr",
    "metrics.eval_loss": "eval_loss",
}
have = [c for c in cols if c in df.columns]
view = df[have].rename(columns=cols)
if "eval_loss" in view.columns:
    view = view.dropna(subset=["eval_loss"]).sort_values("eval_loss")
print(view.to_string(index=False))

In [ ]:
# read the saved generations side by side (the tiebreaker)
import json
import os

for run in ["baseline", "more_epochs", "rank32"]:
    p = f"outputs/adapters/{run}/sample_generations.json"
    if not os.path.exists(p):
        continue
    print(f"\n{'='*60}\n{run}\n{'='*60}")
    for s in json.load(open(p)):
        print("Q:", s["prompt"][:70])
        print("A:", s["response"][:180], "\n")

### How to pick the winner
1. **Correctness first** — are the finance facts and explanations right? A confidently-wrong run loses even with lower eval_loss.
2. **Honesty behavior** — does it refuse to predict prices / guarantee returns instead of hallucinating?
3. **Tiebreak on simplicity** — if two tie, pick the cheaper (lower r, fewer epochs).

Set `WINNER` below to your chosen run.

## 11. Merge the WINNER (base + adapter) → full model

Merge at **full precision** (not 4-bit — merging into 4-bit is lossy). Free the training model, reload the
base in bf16/fp16, attach the winning adapter, `merge_and_unload()`.

In [ ]:
WINNER = "baseline"  # <-- set to your chosen run

import gc

from peft import PeftModel
from transformers import AutoModelForCausalLM

try:
    del model, trainer
except NameError:
    pass
gc.collect()
torch.cuda.empty_cache()

adapter_dir = f"outputs/adapters/{WINNER}/final_adapter"
base_fp = AutoModelForCausalLM.from_pretrained(
    CONFIG["base_model"], torch_dtype=COMPUTE_DTYPE, device_map="auto", trust_remote_code=True
)
merged = PeftModel.from_pretrained(base_fp, adapter_dir).merge_and_unload()

merged_dir = f"outputs/merged/{WINNER}"
merged.save_pretrained(merged_dir, safe_serialization=True)
tokenizer.save_pretrained(merged_dir)
print(">> merged model saved to:", merged_dir)

# sanity check the merged model (normal + honesty)
_m = merged.eval()
for q in ["What is an ETF versus a mutual fund?", "Which stock will double next month?"]:
    _msgs = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": q}]
    _t = tokenizer.apply_chat_template(_msgs, tokenize=False, add_generation_prompt=True,enable_thinking=False)
    _i = tokenizer(_t, return_tensors="pt").to(_m.device)
    with torch.no_grad():
        _o = _m.generate(**_i, max_new_tokens=80, do_sample=False,
                         pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id)
    print("\nQ:", q)
    print("A:", tokenizer.decode(_o[0][_i["input_ids"].shape[1]:], skip_special_tokens=True).strip())

## 12. Push the merged model to the Hugging Face Hub

The Hub is your **model registry**. Needs a token with **write** access: https://huggingface.co/settings/tokens

In [ ]:
from huggingface_hub import login, whoami

login()  # uses HF_TOKEN env if set, else prompts
HF_USERNAME = whoami()["name"]
print("logged in as:", HF_USERNAME)

In [ ]:
from huggingface_hub import create_repo, list_repo_files, upload_folder

REPO_ID = f"{HF_USERNAME}/finbot-qwen3-1.7b-{WINNER}"   # rename if you like
create_repo(REPO_ID, repo_type="model", private=True, exist_ok=True)

card = f"""---
license: apache-2.0
base_model: {CONFIG["base_model"]}
tags: [finance, qlora, peft, fine-tuned]
---

# finbot — {CONFIG["base_model"]} fine-tuned for finance education ({WINNER})

QLoRA fine-tune of `{CONFIG["base_model"]}` (merged, full precision), built as part of an
end-to-end MLOps pipeline (data curation → fine-tuning → evaluation gate → in-cluster
serving → monitoring).

- LoRA r={CONFIG["lora_r"]}, alpha={CONFIG["lora_alpha"]}
- Data: curated `gbharti/finance-alpaca` + hand-written honesty examples (Day 1). Data version: `{data_version()}`.

## Intended use
Educational explanations of general finance concepts (IRAs, ETFs, compound interest,
diversification, etc.). It is **not** a source of personalized financial advice.

## Honest by design
Trained to decline predicting prices, guaranteeing returns, or giving personalized
advice. **Not a licensed financial advisor.**

## Status & known limitations (v1 — baseline)
This is a **v1 baseline release**. Current quality is **average / acceptable for an
educational demo, but not production-grade**:
- Answers are generally coherent and the honesty guardrails work well, but the model
  can be **verbose** and is **not consistently accurate on finer factual details**.
- These characteristics trace primarily to the **training data**: `finance-alpaca` is
  largely sourced from public finance forums, whose style is conversational and whose
  factual quality is uneven. As expected in supervised fine-tuning, the model reflects
  the distribution of its training data — model quality is bounded by data quality.

## Planned improvements (future v2)
A future revision will focus specifically on **data quality**, which is the highest-impact
lever for this model:
- Stricter answer-quality curation (removing rambling, first-person, and low-signal
  responses; filtering answers containing unverifiable claims or links).
- A smaller, cleaner, higher-signal training set (quality over quantity).
- Re-tuning of training and decoding settings based on evaluation results.

The v1 release deliberately prioritizes a **complete, working, observable pipeline** over
peak model accuracy; model-quality improvements are scheduled as a focused follow-up once
the platform (serving, evaluation, and monitoring) is in place.

## Recommended inference settings
Use mild anti-repetition decoding for best results:
`temperature=0.3, top_p=0.9, repetition_penalty=1.2, no_repeat_ngram_size=4`.

## System prompt used in training
> {SYSTEM}
"""
with open(f"{merged_dir}/README.md", "w") as f:
    f.write(card)

upload_folder(repo_id=REPO_ID, folder_path=merged_dir, repo_type="model",
              commit_message=f"merged winner: {WINNER} (v1 baseline)")
print(f">> pushed: https://huggingface.co/{REPO_ID}")

# verify weights actually landed before you delete anything
files = list_repo_files(REPO_ID)
assert any(f.endswith(".safetensors") for f in files), "no weights on the Hub!"
print(">> verified: weights present on the Hub")

## 13. Export GGUF (for CPU serving on Day 4)

Your platform serves the model on **CPU via llama.cpp**, which needs a **GGUF** file. We build llama.cpp,
convert the merged model to GGUF, quantize it (`Q4_K_M`), and push it to its own Hub repo. This is the exact
file Day 4 loads in-cluster.

In [ ]:
# build llama.cpp once (convert script + quantizer)
!git clone --depth 1 https://github.com/ggml-org/llama.cpp
!cd llama.cpp && pip -q install -r requirements.txt && cmake -B build && cmake --build build --config Release -j

In [ ]:
import subprocess
from pathlib import Path

prefix = f"outputs/gguf/finbot-qwen3-1.7b-{WINNER}"
Path("outputs/gguf").mkdir(parents=True, exist_ok=True)
bf16_gguf = f"{prefix}-bf16.gguf"
quant_gguf = f"{prefix}-{CONFIG['gguf_quant']}.gguf"

# 1) merged HF model (bf16) -> GGUF (bf16, matching the weights)
subprocess.run(["python", "llama.cpp/convert_hf_to_gguf.py", merged_dir,
                "--outfile", bf16_gguf, "--outtype", "bf16"], check=True)
# 2) quantize to the served format
subprocess.run(["./llama.cpp/build/bin/llama-quantize", bf16_gguf, quant_gguf, CONFIG["gguf_quant"]], check=True)
print(">> quantized GGUF:", quant_gguf)

In [ ]:
from huggingface_hub import HfApi

GGUF_REPO = f"{HF_USERNAME}/finbot-qwen3-1.7b-gguf"
api = HfApi()
api.create_repo(GGUF_REPO, repo_type="model", private=True, exist_ok=True)
api.upload_file(path_or_fileobj=f"{merged_dir}/README.md", path_in_repo="README.md",
                repo_id=GGUF_REPO, repo_type="model")
api.upload_file(path_or_fileobj=quant_gguf, path_in_repo=Path(quant_gguf).name,
                repo_id=GGUF_REPO, repo_type="model")
print(f">> pushed GGUF: https://huggingface.co/{GGUF_REPO}")
print(">> Day 4 llama.cpp loads:", Path(quant_gguf).name)

## 14. Preserve, then finish

Save what you can't regenerate. If you rented a GPU, **terminate it now** (idle GPUs still bill).

In [ ]:
# freeze the exact environment so a future GPU session installs in one line
%pip freeze > requirements-lock.txt
print("wrote requirements-lock.txt")
print("Keep: outputs/mlruns (run history), requirements-lock.txt. Models already on the Hub.")
print("If this was a rented GPU: TERMINATE it now.")

## ✅ Done

You have:
- Tracked run(s) in `outputs/mlruns` (view with `mlflow ui --backend-store-uri file:outputs/mlruns`)
- Lightweight adapter(s) + saved generations
- The **merged winner** on the HF Hub (full precision)
- A **quantized GGUF** on the HF Hub — the artifact Day 4 serves on CPU via llama.cpp

**Next (Day 3):** put the **quantized GGUF** behind an **eval gate** (golden-set scoring + pass/fail + a
semver tag) before it's allowed to deploy — because quantization slightly changes behaviour, we test the
exact thing users will hit.